## **Assignment - 3**

1.   Name - Atish Kadam
2.   Roll No - CS25MTECH14003

**Importing Library**

In [ ]:
import numpy as np
np.set_printoptions(precision=10, suppress=True)

**Load Linear Programming Instance**

In [ ]:
def check_testcase(csv_path: str):
    data = np.genfromtxt(csv_path, delimiter=",", dtype=float)
    n = data.shape[1] - 1
    m = data.shape[0] - 1
    c = data[0, :n]
    A = data[1:, :n]
    b = data[1:, n]
    return A, b, c, m, n

**Active Constraints Identification**

In [ ]:
def activeness(A, b, z, tol=1e-9):
    return np.where(np.abs(b - A @ z) <= tol)[0]

**Calculate Nullspace**

In [ ]:
def check_nullspace(M, tol=1e-12):
    if M.size == 0:
        return np.eye(M.shape[1])
    U, s, Vt = np.linalg.svd(M, full_matrices=True)
    r = (s > tol).sum()
    return Vt[r:].T

**Find feasible point**

In [ ]:
def find_point(A, b, verbose=True, tol=1e-9):
    m, n = A.shape
    if np.all(b >= -tol):
        z0 = np.zeros(n)
        if verbose:
            print("Origin is feasible!")
            print(f"Initial feasible point: {z0}")
        return z0, True

    if verbose:
        print(f"Origin not feasible (some b_i < 0)")
        print("Attempting to find feasible point...")

    try:
        z0 = np.linalg.lstsq(A, b, rcond=None)[0]

        for attempt in range(100):
            violations = A @ z0 - b
            if np.all(violations <= tol):
                if verbose:
                    print(f"Found feasible point after {attempt+1} attempts")
                    print(f"Initial feasible point: {z0}")
                return z0, True

            # Move towards feasibility
            max_viol_idx = np.argmax(violations)
            if violations[max_viol_idx] > tol:
                # Move in direction that reduces violation
                normal = A[max_viol_idx, :]
                step = (violations[max_viol_idx] + tol) / (np.linalg.norm(normal)**2 + 1e-10)
                z0 = z0 - step * normal

        # Final check
        if np.all(A @ z0 <= b + tol):
            if verbose:
                print(f"Found feasible point")
                print(f"Initial feasible point: {z0}")
            return z0, True

    except:
        pass

    if verbose:
        print("Could not find initial feasible point")
        print("Problem may be infeasible")

    return None, False

**Phase - 2**

In [ ]:
def next_step(A, b, c, z0, verbose=True, tol=1e-9, max_iter=5000):
    m, n = A.shape

    z = z0.copy()
    path = [z.copy()]
    vals = [float(c @ z)]

    visited_states = set()
    degenerate_count = 0
    last_obj_value = c @ z

    # Move to first vertex
    if verbose:
        print("\nMoving to first vertex...")

    for _ in range(5 * (m + n)):
        I = activeness(A, b, z, tol)
        if len(I) >= n:
            if verbose:
                print(f"First vertex: z = {z}, f = {c @ z:.10f}")
            break

        N = check_nullspace(A[I, :], tol) if len(I) > 0 else np.eye(n)
        d = N @ (N.T @ c)
        if np.linalg.norm(d) < tol:
            if N.shape[1] > 0:
                d = N[:, 0]
            else:
                break

        Ad = A @ d
        slack = b - A @ z
        alpha, j_in = np.inf, -1
        for j in range(m):
            if Ad[j] > tol:
                step = slack[j] / Ad[j]
                if step < alpha:
                    alpha = step
                    j_in = j

        if not np.isfinite(alpha):
            if verbose:
                print("\nUNBOUNDED: objective → +∞")
            return None, path, vals

        z = z + alpha * d
        path.append(z.copy())
        vals.append(float(c @ z))

    # Edge walking with cycle detection
    if verbose:
        print("\nEdge walking to optimal vertex...")

    for it in range(max_iter):
        I = activeness(A, b, z, tol)

        state_sig = (tuple(np.round(z, 8)), tuple(sorted(I)))

        if state_sig in visited_states:
            if verbose:
                print(f"\nCYCLE DETECTED at iteration {it+1}")
                print(f"Current vertex: {z}")
                print(f"Optimal value: {c @ z:.10f}")
            break
        visited_states.add(state_sig)

        if len(I) >= n:
            I_sel = []
            R = None
            I_sorted = sorted(I)

            for r in I_sorted:
                if R is None:
                    R = A[[r], :]
                    I_sel = [r]
                else:
                    candidate = np.vstack([R, A[r, :]])
                    if np.linalg.matrix_rank(candidate, tol) > np.linalg.matrix_rank(R, tol):
                        R = candidate
                        I_sel.append(r)
                        if len(I_sel) == n:
                            break

            if len(I_sel) < n:
                if verbose:
                    print("\nCould not find n linearly independent constraints.")
                break

            A_I = A[I_sel, :]

            try:
                lam = np.linalg.solve(A_I.T, c)
            except np.linalg.LinAlgError:
                if verbose:
                    print("\nSingular matrix encountered.")
                break

            if np.all(lam >= -tol):
                if verbose:
                    print("\nOptimal vertex reached.")
                break

            neg_indices = [i for i, val in enumerate(lam) if val < -tol]
            if not neg_indices:
                if verbose:
                    print("\nOptimal vertex reached.")
                break
            k = neg_indices[0]

            e = np.zeros(len(I_sel))
            e[k] = 1.0
            d = -np.linalg.solve(A_I, e)
        else:
            N = check_nullspace(A[I, :], tol)
            if N.size == 0:
                if verbose:
                    print("\nEmpty nullspace.")
                break
            d = N @ (N.T @ c)
            if np.linalg.norm(d) < tol:
                if verbose:
                    print("\nNo improving direction found.")
                break

        Ad = A @ d
        slack = b - A @ z

        blocking_candidates = []
        for j in range(m):
            if Ad[j] > tol:
                step = slack[j] / Ad[j]
                blocking_candidates.append((step, j))

        if not blocking_candidates:
            if verbose:
                print("\nUNBOUNDED: objective → +∞")
            return None, path, vals

        blocking_candidates.sort(key=lambda x: (x[0], x[1]))
        alpha, j_in = blocking_candidates[0]

        if np.abs(alpha) < tol:
            degenerate_count += 1
            if verbose and degenerate_count == 1:
                print(f"\nDEGENERATE pivot detected")

        z = z + alpha * d
        new_obj = c @ z

        if np.abs(new_obj - last_obj_value) < tol:
            degenerate_count += 1
            if degenerate_count > 20:
                if verbose:
                    print(f"\nToo many degenerate pivots. Stopping.")
                    print(f"Best vertex found: {z}, f = {new_obj:.10f}")
                break
        else:
            degenerate_count = 0
            last_obj_value = new_obj

        path.append(z.copy())
        vals.append(float(new_obj))

        if verbose and it < 20:  # Only print first 20 iterations
            print(f"[Iter {it+1}] Entered constraint {j_in}; f = {new_obj:.10f}")
        elif verbose and it == 20:
            print("further iterations suppressed")

    return z, path, vals

**Main Algorithm**

In [ ]:
def find_objective(csv_path: str, verbose=True, tol=1e-9, max_iter=5000):
    # Load problem (Assignment 3 format)
    A, b, c, m, n = check_testcase(csv_path)

    if verbose:
        print("\nInitial Problem Setup:\n")
        print(f"Variables (n):   {n}")
        print(f"Constraints (m): {m}")
        print(f"Cost Vector (c): {c}")
        print("Constraint Matrix (A):")
        print(A)
        print(f"Constraint RHS (b): {b}\n\n")

    # Phase 1: Find initial feasible point
    z0, feasible = find_point(A, b, verbose, tol)

    if not feasible:
        if verbose:
            print("\n" + "=" * 60)
            print("RESULT: PROBLEM IS INFEASIBLE")
            print("=" * 60)
        return None, [], []

    # Phase 2: Optimize
    z_opt, path, vals = next_step(A, b, c, z0, verbose, tol, max_iter)

    # Final output
    if verbose:
        if z_opt is not None:
            print(f"Initial feasible point: {z0}")
            print(f"Optimal vertex: {z_opt}")
            print(f"Optimal value: {c @ z_opt:.10f}")
        else:
            print("Problem is UNBOUNDED")

        print(f"\nTotal vertices visited: {len(path)}")

        if len(path) <= 20:
            print("\nSequence of vertices and objective values:")
            for i, (zi, fi) in enumerate(zip(path, vals), start=1):
                print(f"{i}. {zi.tolist()}   f = {fi:.10f}")
        else:
            print("\nFirst 10 vertices:")
            for i in range(min(10, len(path))):
                print(f"{i+1}. {path[i].tolist()}   f = {vals[i]:.10f}")
            print(f"\n... ({len(path) - 20} vertices omitted) ...\n")
            print("Last 10 vertices:")
            for i in range(max(10, len(path)-10), len(path)):
                print(f"{i+1}. {path[i].tolist()}   f = {vals[i]:.10f}")
        return None

    return z_opt, path, vals

**Run Solver**

In [ ]:
find_objective("/content/Testcase4.csv", verbose=True)


Initial Problem Setup:

Variables (n):   2
Constraints (m): 5
Cost Vector (c): [3. 7.]
Constraint Matrix (A):
[[ 1.  0.]
 [ 0.  1.]
 [ 2.  1.]
 [ 1.  2.]
 [-1. -1.]]
Constraint RHS (b): [ 4.  4.  8.  4. -1.]


Origin not feasible (some b_i < 0)
Attempting to find feasible point...
Found feasible point after 2 attempts
Initial feasible point: [2.7666666665 0.6166666663]

Moving to first vertex...
First vertex: z = [-1.999999999  2.999999999], f = 14.9999999961

Edge walking to optimal vertex...

Optimal vertex reached.
Initial feasible point: [2.7666666665 0.6166666663]
Optimal vertex: [-1.999999999  2.999999999]
Optimal value: 14.9999999961

Total vertices visited: 2

Sequence of vertices and objective values:
1. [2.766666666469667, 0.6166666662726669]   f = 12.6166666633
2. [-1.9999999990150021, 2.9999999990150026]   f = 14.9999999961
